# 02 · Perfil dos dados · Olist
Análise exploratória que fundamenta as regras de negócio aplicadas na silver e a calibração dos geradores sintéticos.

### 1. Peso de cada categoria
**Decisão que fundamenta:** o de-para de categorias (`de_para_categorias.csv`).

**Achado:** roupas e calçados somam só 611 itens (0,5%). O mix adotado inclui moda, acessórios, beleza, casa e esporte, totalizando 43.034 itens (38% do Olist).

In [0]:
-- Databricks SQL
SELECT p.product_category_name                                   AS categoria,
       COUNT(*)                                                  AS itens,
       ROUND(SUM(CAST(i.price AS DECIMAL(12,2))), 2)             AS receita,
       ROUND(100 * SUM(CAST(i.price AS DECIMAL(12,2)))
             / SUM(SUM(CAST(i.price AS DECIMAL(12,2)))) OVER (), 2) AS pct_receita
FROM horizonte.bronze.olist_order_items i
LEFT JOIN horizonte.bronze.olist_products p USING (product_id)
GROUP BY 1
ORDER BY receita DESC;

### 2. Status dos pedidos
**Decisão que fundamenta:** a regra de pedido válido.

**Achado:** 97% dos pedidos estão `delivered`. São excluídos `canceled` (625) e `unavailable` (609). Os demais status são vendas em andamento e entram, pois a venda é contada na data da compra.

In [0]:
SELECT order_status, COUNT(*) AS pedidos
FROM horizonte.bronze.olist_orders
GROUP BY 1
ORDER BY 2 DESC;

### 3. Período coberto
**Decisão que fundamenta:** o deslocamento de datas.

**Achado:** compras de 04/09/2016 a 17/10/2018. O deslocamento de 418 semanas (2.926 dias) alinha a Black Friday de 2017 com a de 2025 e preserva o dia da semana.

In [0]:
SELECT MIN(CAST(order_purchase_timestamp AS TIMESTAMP)) AS primeira_compra,
       MAX(CAST(order_purchase_timestamp AS TIMESTAMP)) AS ultima_compra
FROM horizonte.bronze.olist_orders;

### 4. Volume mensal
**Decisão que fundamenta:** a janela útil (`periodo_completo`).

**Achados:**
- Set–dez/2016 e set–out/2018 são quase vazios (349 pedidos no total); nov/2016 não tem nenhum pedido. Janela útil: jan/2017 a ago/2018.
- Nov/2017 tem 7.544 pedidos, +63% sobre outubro: efeito da Black Friday, a única presente nos dados.
- 2017 mostra crescimento da própria plataforma Olist (de 800 para cerca de 4.600 pedidos/mês), não sazonalidade. Ponto de atenção para o modelo de previsão.

In [0]:
SELECT date_trunc('MONTH', CAST(order_purchase_timestamp AS TIMESTAMP)) AS mes,
       COUNT(*) AS pedidos
FROM horizonte.bronze.olist_orders
GROUP BY 1 ORDER BY 1;

### 5. Pedidos sem itens
**Decisão que fundamenta:** a regra de pedido válido exige ao menos um item.

**Achado:** 775 pedidos não têm itens; 767 são cancelados ou indisponíveis. Os 8 restantes também são excluídos. Atenção: 461 pedidos cancelados **têm** itens, por isso a exclusão é feita pelo status, não só pelo join.

In [0]:
-- funciona igual nos dois motores, mudando só o nome das tabelas
SELECT o.order_status, COUNT(*) AS pedidos_sem_itens
FROM horizonte.bronze.olist_orders o
LEFT JOIN horizonte.bronze.olist_order_items i USING (order_id)
WHERE i.order_id IS NULL
GROUP BY 1;

### 6. Notas e comentários das avaliações
**Decisão que fundamenta:** a calibração do gerador de avaliações do site (D25): distribuição das notas, chance de comentário e de título por nota.

**Achados:**
- 99.224 avaliações de 98.673 pedidos (99,2% dos pedidos): no Olist a avaliação é uma pesquisa de satisfação enviada a todo cliente, e não uma avaliação de produto voluntária. Por isso a **taxa de avaliação** do site é premissa do squad (10% dos itens), não vem daqui.
- Notas em forma de J: 57,8% dão 5 e 11,5% dão 1; as notas intermediárias são raras.
- Quanto pior a nota, mais a pessoa escreve: 76,5% das notas 1 têm comentário, contra 35,8% das notas 5. Só 11,7% têm título.
- Comentários curtos e mais longos nas notas baixas: mediana de 89 caracteres na nota 1 e 42 na nota 5 (máximo de 208, limite do formulário).
- Os textos não são reaproveitados: tratam do pedido e da entrega, não do produto, e não há SKU do catálogo para ligá-los (D12).

In [0]:
SELECT review_score AS nota,
       COUNT(*) AS avaliacoes,
       ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_avaliacoes,
       ROUND(100 * AVG(CASE WHEN trim(coalesce(review_comment_message, '')) <> '' THEN 1 ELSE 0 END), 1) AS pct_com_comentario,
       ROUND(100 * AVG(CASE WHEN trim(coalesce(review_comment_title, '')) <> '' THEN 1 ELSE 0 END), 1) AS pct_com_titulo,
       percentile_approx(CASE WHEN trim(coalesce(review_comment_message, '')) <> '' THEN length(trim(review_comment_message)) END, 0.5) AS mediana_caracteres
FROM horizonte.bronze.olist_order_reviews
GROUP BY review_score
ORDER BY nota;

### 7. Prazo até a avaliação
**Decisão que fundamenta:** a data das avaliações geradas (quantos dias depois da compra a pessoa avalia).

**Achado:** entre a compra e a resposta da pesquisa passam [mediana] dias (10% em até [p10] dias; 90% em até [p90]), considerando só pedidos entregues. Usa-se `review_answer_timestamp` (resposta do cliente), não `review_creation_date` (envio da pesquisa, sempre à meia-noite).

In [0]:
WITH prazo AS (
  SELECT datediff(to_date(r.review_answer_timestamp), to_date(o.order_purchase_timestamp)) AS dias
  FROM horizonte.bronze.olist_order_reviews r
  JOIN horizonte.bronze.olist_orders o ON o.order_id = r.order_id
  WHERE o.order_status = 'delivered'
)
SELECT COUNT(*) AS avaliacoes,
       percentile_approx(dias, 0.1) AS p10,
       percentile_approx(dias, 0.5) AS mediana,
       percentile_approx(dias, 0.9) AS p90
FROM prazo;